# 05 — Agente de compras, herramientas y memoria

Un único Qwen2.5-7B-Instruct coordina nueve herramientas. Nuestro código conserva presupuesto, preferencias, carrito y cotización; valida cantidades, existencias, consentimiento y registro del pedido. El modelo no puede confirmar por sí solo: la tienda y las reglas de control verifican cada acción.


In [ ]:
%pip -q install -r requirements-lab4-modular.txt

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
from IPython.display import display
from lab4_asistente.runtime import cargar_sistema


## Cargar los componentes

In [ ]:
COMP = cargar_sistema(cargar_llm=True)
SIS = COMP["sistema"]
print("Dispositivo:", COMP["dispositivo"])
print("Ubicación por componente:", COMP["dispositivos"])
print("Carga inicial por componente:", COMP["cargas"])
print("Herramientas disponibles:", len(__import__("lab4_asistente.herramientas", fromlist=["ESQUEMAS"]).ESQUEMAS))


## Interacción por turnos

In [ ]:
SIS.reiniciar()
r1 = SIS.agente.turno("Busco una mochila para portátil negra, máximo 40 euros")
print(r1["respuesta"])
print("Herramientas:", [(o["herramienta"], o["resultado"].get("ok")) for o in r1["operaciones"]])
r2 = SIS.agente.turno("¿Qué opciones encontraste?")
print(r2["respuesta"])
print("Estado persistente:")
print(SIS.h.resumen_estado())


## Verificación del camino crítico de compra

In [ ]:
# El agente debe presentar una cotización antes de poder registrar un pedido.
# La confirmación explícita se ejecuta y valida en código; la frase del LLM nunca basta.
print("Estado de cotización:", SIS.h.estado.cotizacion)
print("¿Cotización mostrada?:", SIS.h.estado.cotizacion_mostrada)


## Correcciones incorporadas respecto al prototipo anterior

- `agregar_al_carrito` fija la cantidad total deseada; una llamada repetida no suma unidades.
- Repetir la misma cantidad no cambia la versión del carrito ni invalida la cotización.
- Si el usuario confirma una cotización vigente, el código ejecuta la herramienta de confirmación.
- El agente reemplaza cualquier afirmación de pedido confirmado si no existe un resultado exitoso de la tienda.